<!-- HTML file automatically generated from DocOnce source (https://github.com/doconce/doconce/)
doconce format html week15.do.txt --no_mako -->
<!-- dom:TITLE: Quantum Computing and Quantum Machine Learning -->

# Quantum Computing and Quantum Machine Learning
**Morten Hjorth-Jensen**, Department of Physics, University of Oslo

Date: **April 29, 2026**

## Plan for the week of April 27–May 2

1. Discussion of the **QAOA algorithm** with repetition from last week

2. Parametrized quantum circuits (PQC) and Variational Quantum Circuits (VQCs)

3. **Quantum Neural Networks (QNNs):** mathematical structure, QFI, TDVP, barren plateaus

4. **Variational Quantum Eigensolver (VQE):** variational ansatz, UCC, connections to many-body theory

5. **Quantum Approximate Optimization Algorithm (QAOA):** adiabatic motivation, MaxCut, landscape

6. **Quantum Support Vector Machines (QSVM):** quantum kernels, fidelity, RKHS

7. **HHL Algorithm:** solving linear systems, QPE, Green's functions

8. Grand unification: Geometry + Dynamics + Operators


## What is Quantum Machine Learning?

Quantum Machine Learning (QML) integrates quantum computing with
machine learning algorithms to exploit quantum advantages. It explores
how quantum computing can enhance classical machine learning.

**Motivation:**

1. High-dimensional Hilbert spaces for better feature representation.

2. Quantum parallelism for faster computation.

3. Quantum entanglement for richer data encoding.

## Quantum Speedups in ML
Why Quantum?
1. **Quantum Parallelism:** Process multiple states simultaneously.

2. **Quantum Entanglement:** Correlated states for richer information.

3. **Quantum Interference:** Constructive and destructive interference to enhance solutions.

## Challenges in Quantum Machine Learning

**Quantum Hardware Limitations:**

1. Noisy Intermediate-Scale Quantum (NISQ) devices.

2. Decoherence and limited qubit coherence times.

**Data Encoding:**

1. Efficient embedding of classical data into quantum states.

**Scalability:**

1. Difficult to scale circuits to large datasets.

---
## Quantum Neural Networks: Mathematical Framework

*This section develops the rigorous operator-theoretic foundation of QNNs, connecting to geometry, dynamics, and gradient methods.*

## Quantum neural network

Another variation is the quantum variational classifier, sometimes
called a quantum neural network (to be discussed below).  Instead of precomputing a fixed
kernel, one trains a parameterized quantum circuit to output labels.
Interestingly, Schuld (2021) shows that variational quantum models,
when trained by minimizing a loss, are mathematically equivalent to
kernel machines with a particular kernel determined by the circuit .
In fact, one can often find a kernel SVM that matches or outperforms
the variational model.  In practice, one can combine these: use a
trainable quantum embedding $U(\boldsymbol{x};\Theta)$ with tunable
parameters $\Theta$, and optimize $\Theta$ to maximize the SVM
classification accuracy.  This is called a quantum kernel learning
approach.

## Quantum Neural Networks and Variational Circuits

The Variational Quantum Algorithm (VQA) is a 
Variational Quantum Circuit (VQC), that is  a quantum circuit with tunable
parameters and which is trained using a classical optimizer.  In practice, a
VQC (also called a Parameterized Quantum Circuit (PQC)) is used as a
Quantum Neural Network (QNN): data are encoded into quantum states, a
parameterized circuit is applied, and measurements yield outputs.
For example, Abbas et al. showed that certain QNNs can exhibit higher
effective dimension (and thus capacity to generalize) than comparable
classical networks , suggesting a potential quantum advantage.

Below we develop the mathematical foundations
(state preparation, parameterized unitaries, measurement), discuss
optimization and training challenges, and work through practical code
examples using PennyLane.

## Variational Quantum Circuits

Variational Quantum Algorithms (VQAs) are hybrid schemes where a
quantum circuit with adjustable parameters is trained by a classical
optimizer .  In this framework, a Variational Quantum Circuit (VQC)
typically has three parts : (i) a state preparation or feature map
that encodes classical input $\mathbf{x}$ into a quantum state; (ii) a
parameterized circuit $W(\boldsymbol\Theta)$ (often called the ansatz)
that depends on trainable parameters $\boldsymbol\Theta$; and (iii) a
measurement that extracts a classical output from the final quantum
state.

## Setting up a VQC

Given an input vector $\mathbf{x}=(x_1,\dots,x_n)$, we prepare the initial state

$$
\vert \psi_{\rm in}\rangle = U(\mathbf{x})|0\rangle^{\otimes n},
$$

where $U(\mathbf{x})$ is a unitary (possibly composed of rotations)
that depends on the data.  We then apply the variational circuit
$W(\boldsymbol\Theta)$, often built as a product of layers
$V_j(\Theta_j)$, so that the final state is

$$
\vert \Psi(\mathbf{x};\boldsymbol\Theta)\rangle = W(\boldsymbol\Theta)U(\mathbf{x})|0\rangle^{\otimes n}.
$$

For instance, one common ansatz is the hardware-efficient circuit:
layers of parameterized single-qubit rotations and entangling gates
(like CNOTs) repeated several times.  The structure of
$W(\boldsymbol\Theta)$ can dramatically affect the circuit’s
expressivity and trainability.

## Outputs

To produce a scalar or vector output, we measure one or more
observables $\hat B_k$ on the final state.  The network’s output is
given by the expectation values:

$$
f_k(\mathbf{x};\boldsymbol\Theta) = \langle \Psi(\mathbf{x};\boldsymbol\Theta) | \hat B_k | \Psi(\mathbf{x};\boldsymbol\Theta)\rangle.
$$

Equivalently, with

$$
\vert \Psi(\mathbf{x};\boldsymbol\Theta)\rangle = W(\boldsymbol\Theta)U(\mathbf{x})|0\rangle,
$$

one has

$$
f_k(\mathbf{x};\boldsymbol\Theta) = \langle 0|U(\mathbf{x})^\dagger W(\boldsymbol\Theta)^\dagger\hat B_k W(\boldsymbol\Theta) U(\mathbf{x})|0\rangle.
$$

Commonly $\hat B$ is a Pauli operator (e.g. $Z$ on one qubit).  In practice one runs many shots on quantum hardware or simulates this circuit classically to estimate $\langle \hat B_k\rangle$ .

## Short summary

In summary, a variational quantum model
$f(\mathbf{x};\boldsymbol\Theta)$ maps inputs to outputs via the
hybrid quantum-classical procedure.  During training, the classical
optimizer adjusts $\boldsymbol\Theta$ (e.g. by gradient descent) to
minimize a cost function (like mean-squared error) defined on a
dataset.  Because the mapping is inherently quantum, these models can,
in principle, harness the high-dimensional Hilbert space for richer
representations.  (However, unlike classical deep nets, VQCs may face
unique challenges such as gradient vanishing, which we discuss later.)

## QNN as Variational Quantum State

The most compact mathematical description of a QNN is:

$$
|\psi(x,\theta)\rangle = U(\theta)\,U(x)\,|0\rangle
$$

$$
f(x,\theta) = \langle \psi(x,\theta)\,|\,O\,|\,\psi(x,\theta)\rangle
$$

where:

- $U(x)$: **feature map** — encodes classical data into a quantum state
- $U(\theta)$: **variational ansatz** — trainable unitary
- $O$: **observable** — defines the output

### Feature Map as Hamiltonian Evolution

A common choice encodes data as time evolution under a data-dependent Hamiltonian:

$$
U(x) = e^{-iH(x)}, \qquad H(x) = \sum_i x_i Z_i + \sum_{i<j} x_i x_j Z_i Z_j
$$

This embeds the data as coupling constants of a spin Hamiltonian — the circuit implements quantum time evolution.

### Pauli Expansion of the Ansatz

The variational unitary is generated by a Pauli Hamiltonian:

$$
U(\theta) = e^{-iH(\theta)}, \qquad H(\theta) = \sum_\alpha \theta_\alpha P_\alpha,
\qquad P_\alpha \in \{I,X,Y,Z\}^{\otimes n}
$$

This is directly analogous to a many-body Hamiltonian with tunable coupling constants.


## BCH Expansion and Effective Operator Structure

The Baker–Campbell–Hausdorff expansion reveals what the circuit computes:

$$
U^\dagger O U = O + i[H,O] + \frac{i^2}{2}[H,[H,O]] + \cdots
$$

This generates a **hierarchy of correlations**, directly analogous to coupled-cluster expansions in nuclear and quantum chemistry. The effective observable is:

$$
O_{\mathrm{eff}} = U^\dagger O U
$$

- Circuit depth $\rightarrow$ higher-order commutators
- Deep circuits encode highly nonlinear transformations


## Mathematical example

For concreteness, consider a 2-qubit circuit.  A simple encoding is

$$
U(\mathbf{x})=R_x(x_1)\otimes R_x(x_2),
$$

and a variational layer is

$$
V(\boldsymbol\Theta)=R_y(\Theta_1)\otimes R_y(\Theta_2)\mathrm{CNOT}(0,1),
$$

(apply $R_y$ on each qubit then entangle).  After
applying $W(\boldsymbol\Theta)=V(\boldsymbol\Theta)$ to $|00\rangle$,
we measure $\hat B=Z\otimes I$ on qubit 0.  The output is

$$
f(\mathbf{x};\boldsymbol\Theta) = \langle 00|U(\mathbf{x})^\dagger V(\boldsymbol\Theta)^\dagger (Z\otimes I)V(\boldsymbol\Theta)U(\mathbf{x})|00\rangle.
$$

This $f(x;\Theta)$ is then compared to the target in a cost function for optimization.

## Key elements

A VQC is a quantum circuit with trainable parameters acting on a
quantum state; it is central to near-term QML (hybrid
quantum-classical).  Data encoding and ansatz design determine a
VQC’s expressivity.  Simple encodings use rotations (e.g. $R_x(x_i)$)
on each qubit , while more complex feature maps may exploit
entanglement.  The circuit output is obtained via expectation values
of observables (e.g. Pauli-Z), yielding a differentiable function
$f(\mathbf{x};\boldsymbol\Theta)$ .

## Test yourself exercises

1. Compute the state $|\Psi(\mathbf{x};\boldsymbol\Theta)\rangle$ explicitly for a 1-qubit VQC with $U(x)=R_x(x)$ and $W(\Theta)=R_y(\Theta)$. What is $\langle Z\rangle$ as a function of $x,\Theta$?

2. Draw (or describe) a hardware-efficient ansatz for 3 qubits with 2 layers of rotations and CNOTs. How many parameters does it have?

For the above ansatz, derive the effect of each layer on the state’s parameters.

## Quantum Neural Networks (QNNs)

Quantum Neural Networks (QNNs) are essentially multi-layer VQCs that
mimic classical neural network architectures .  One can think of each
layer as adding a nonlinear quantum neuron to the network.  A simple
QNN is a sequence of encoding and variational layers.  More structured
architectures also exist, such as Quantum Convolutional Neural
Networks (QCNNs) and Quantum Long-Short Term Memory networks.  In a
QCNN, for example, qubits are entangled in a localized pattern to
mimic convolution and pooling .

## Input Encoding

A crucial aspect of any QNN (as we also saw for QSVMs) is how
classical data $\mathbf{x}\in\mathbb{R}^d$ are embedded into a quantum
state.  Common strategies include:
1. Basis Encoding: Map each bit of $\mathbf{x}$ (or feature) to a qubit state $|0\rangle$ or $|1\rangle$. Simple but limited to binary data.

2. Angle (Amplitude) Encoding: Use rotation gates to encode real values, e.g. $R_x(x_i)$ or $R_y(x_i)$ on qubit $i$.  

3. Amplitude Encoding: Embed $\mathbf{x}$ into the amplitudes of a multi-qubit state (exponentially compact, but requires complex circuits to prepare).

4. Data Re-uploading: Re-encode input at multiple layers interspersed with trainable gates, effectively increasing expressivity.

The choice of feature map affects performance: no single encoding is
best for all tasks.  Often one uses a problem-inspired map or random
feature circuits, then lets the optimizer adjust the ansatz.

## QNN Architecture and Models

A general QNN can be viewed as a parameterized unitary
$U(\mathbf{x},\boldsymbol\Theta)$ acting on $n$ qubits, followed by
measurements.  Fig. 2 (placeholder) might depict a generic QNN with
several layers of trainable gates. Each layer can entangle qubits,
building up complexity. The output is then a (classical) vector of
measured values, analogous to the output layer in a classical network.

## A simple feedforward QNN structure

1. Embedding Layer: Convert $\mathbf{x}$ to $|0\rangle^{\otimes n}$ via $U(\mathbf{x})$.

2. Variational Layers: Repeat $L$ blocks of parameterized gates $W(\boldsymbol\Theta^{(l)})$ (each block may act on all or subsets of qubits).

3. Measurement: Measure selected qubits or observables to obtain the output predictions $f(\mathbf{x};\boldsymbol\Theta)$.

## Example

For example, a 2-layer QNN on 2 qubits might apply encoding
$R_x(x_1)\otimes R_x(x_2)$, then apply $W(\Theta^{(1)})$, then again
encoding (or not), then $W(\Theta^{(2)})$, and finally measure. In
classification tasks, one typically assigns a label based on the sign
of $\langle Z\rangle$ or uses multiple measurements for multi-class
outputs.

Notably, even though QNNs operate on exponentially large Hilbert
spaces, their actual power is subject of research. Abbas et
al. introduce the notion of effective dimension and argue that some
QNNs can outperform classical networks in terms of trainability and
generalization .  However, other studies point out that QNNs may
suffer from trainability issues.

## Training output and Cost/Loss-function

Given a QNN with output $f(\mathbf{x};\boldsymbol\Theta)$ (a real
number or vector of real values), one must define a loss function to
train on data. Common choices are the mean squared error (MSE) for
regression or cross-entropy for classification.  For a training set
${\mathbf{x}i,y_i}$, the MSE cost/loss-function is

$$
C(\boldsymbol\Theta) = \frac{1}{N} \sum_{i=1}^N \bigl(f(\mathbf{x}i;\boldsymbol\Theta) - y_i\bigr)^2.
$$

One then computes gradients $\nabla{\boldsymbol\Theta}C$ and updates
parameters via gradient descent or other optimizers.

## Exampe: Variational Classifier

A binary classifier can output
$f(\mathbf{x};\boldsymbol\Theta)=\langle Z_0\rangle$ on qubit 0, and
predict label $+1$ if $f\ge0$, else $-1$.

## Variational Layer Algebra

As a warm-up problem, consider two qubits with single-qubit rotations
$R_y(\alpha)$ on each qubit followed by a CNOT. Show that this
two-qubit gate can create entanglement if $\alpha$ is not a multiple
of $\pi$.  (Hint: apply it to $|00\rangle$ and compute the resulting
state.)  This demonstrates how trainable gates can correlate qubits,
enriching the model.

## Short summary

A QNN is implemented by layering VQCs; it generalizes neural networks
to quantum circuits .  Encoding maps classical features to quantum
states (e.g. via rotation gates ).  The ansatz (variational layers)
defines the network’s expressive power; depth and entanglement matter.
Output is given by expectation(s) of measured observables, which are
compared against targets via a classical loss function.

## Training QNNs and Loss Landscapes

Training a QNN involves optimizing a non-convex quantum circuit cost
function.  Like classical neural networks, one typically uses
gradient-based methods.  However, VQCs have unique features, as listed here.

## Gradient Computation

Gradients $\partial f/\partial\Theta_j$ are obtained using the parameter-shift rule.  For many gates $e^{-i\Theta P/2}$ (with $P$ a Pauli), one can compute

$$
\frac{\partial}{\partial\Theta}\langle B\rangle
= \frac{1}{2}\Bigl[\langle B\rangle_{\Theta+\pi/2} - \langle B\rangle_{\Theta-\pi/2}\Bigr],
$$

where $\langle B\rangle_{\Theta\pm\pi/2}$ are expectation values
evaluated at shifted parameter values.  This formula allows exact
gradients by two circuit evaluations per parameter (independent of
circuit size).  PennyLane automatically applies parameter-shift rule
when you call backward on a QNode .  Optimizers: One can use gradient
descent or more advanced optimizers (Adam, ADAgrad, RMSprop, etc.). PennyLane
provides a qml.GradientDescentOptimizer and others.  Gradients flow
through the classical loss into the quantum circuit via the
parameter-shift trick. In our code examples below we will see
this in action.

## Quantum Fisher Information Matrix

The **Quantum Fisher Information** (QFI) defines the natural metric on the space of variational quantum states:

$$
F_{ij} = 4\,\mathrm{Re}\!\left(
\langle \partial_i\psi|\partial_j\psi\rangle -
\langle \partial_i\psi|\psi\rangle\langle \psi|\partial_j\psi\rangle
\right)
$$

This defines a **Riemannian metric**:

$$
ds^2 = \sum_{ij} F_{ij}\,d\theta_i\,d\theta_j
$$

which measures the distinguishability of nearby quantum states — i.e., the information-geometric distance.

### Natural Gradient Descent

The standard Euclidean gradient ignores the curved geometry of Hilbert space. The **natural gradient** corrects for this:

$$
\dot{\theta} = F^{-1}\nabla C
$$

This is the basis of **Stochastic Reconfiguration (SR)** in VMC and the imaginary-time TDVP (see below).
It respects the quantum geometry and converges faster in practice.


## Time-Dependent Variational Principle (TDVP)

The TDVP gives the optimal projection of the Schrödinger equation onto the variational manifold.
Starting from:

$$
\delta \|(i\partial_t - H)|\psi\rangle\| = 0
$$

Projecting onto the tangent vectors $|\partial_i\psi\rangle$:

$$
\langle \partial_i\psi\,|\,(i\partial_t - H)\,|\,\psi\rangle = 0
$$

gives the **TDVP equation of motion**:

$$
\sum_j F_{ij}\,\dot{\theta}_j = C_i, \qquad
C_i = \mathrm{Im}\langle \partial_i\psi\,|\,H\,|\,\psi\rangle
$$

**Interpretation:** QNN training is a **classical dynamical system** — the parameters $\theta_j$ evolve like generalized coordinates on a Riemannian manifold with metric $F_{ij}$.
This connects QNN optimization directly to quantum dynamics.


## Barren Plateaus

A major challenge is the barren plateau phenomenon .  In deep or
highly entangled circuits, the loss landscape can become extremely
flat: gradients vanish exponentially with system size.  As Anschuetz
and Kiani note, variational models often become untrainable due to
vanishing gradients in deep layers .  Even surprisingly, their work
shows that shallow circuits may still have very few “good” local
minima near the global optimum .  In practice, this means random
initialization of a deep QNN often leads to tiny gradients, stalling
training.  Mitigation Strategies: Researchers propose various remedies
to avoid or alleviate barren plateaus.  Examples include layerwise
training (training a few layers at a time), smart initialization
(e.g. initializing most gates to identity), and ansatz
design (using problem-inspired or shallow circuits to avoid global
entanglement).  Another approach uses local cost functions: measuring
local observables rather than global ones can reduce gradient
concentration.  These strategies are active research areas, but remain
crucial for making QNN training feasible on near-term devices.

## Cost/Loss-landscape visualization

One can imagine the cost/loss function $C(\boldsymbol\Theta)$ over the
parameter space.  Unlike convex classical problems, this landscape may
have many local minima and saddle points.  Barren plateaus correspond
to regions where $\nabla C\approx 0$ almost everywhere.  Even if
plateaus are avoided, poor minima can still trap the optimizer .  In
practice, careful tuning of learning rates and adding small random
noise can help escape shallow minima.

QNN training uses classical optimizers on circuit outputs, with
gradients given by the parameter-shift rule .  Barren plateaus
(vanishing gradients) are a central obstacle in deep circuits .
Mitigation includes shallow ansatz, structured circuits, and smart
initialization.  Always monitor training and consider multiple random
restarts to find good minima.

## Exercises

1. Compute a gradient by hand: For a circuit with one qubit and $f(\Theta)=\langle0|R_y(\Theta)^\dagger Z R_y(\Theta)|0\rangle$, use the parameter-shift rule to compute $df/d\Theta$.

2. Explore barren plateaus: Numerically evaluate $\partial f/\partial\Theta$ for a simple 5-qubit random circuit as depth increases. Observe the trend of gradient norms. What does this suggest?

3. Optimizer effects: Implement a small QNN (2 qubits) and train with both SGD and Adam optimizers. Compare convergence speed.

## Barren Plateaus: Mathematical Origin

The gradient variance scales as:

$$
\mathrm{Var}(\nabla C) \sim \frac{1}{2^n}
$$

**Why?** Random, deep circuits form approximate **unitary 2-designs** — their output distribution approaches the Haar measure on the unitary group.
Expectation values of local observables concentrate exponentially around their mean, making gradients exponentially small.

### Avoiding Barren Plateaus

| Strategy | Mechanism |
|---|---|
| **Local cost functions** | Observables supported on $O(1)$ qubits scale polynomially |
| **Problem-inspired ansätze** | Exploit structure; avoid the random circuit regime |
| **Layerwise training** | Initialize and train one layer at a time |
| **Entanglement control** | Limit entanglement in early training |

### Entanglement and Expressivity

Entanglement entropy of a subsystem $A$:

$$
S(\rho_A) = -\mathrm{Tr}(\rho_A \log \rho_A)
$$

Three regimes:
- **Low entanglement** → classical-like, efficient simulation, limited expressivity
- **Moderate entanglement** → optimal learning regime
- **High entanglement** → concentration of measure, barren plateaus

### Quantum Neural Tangent Kernel

In the **linearized training regime**, the QNN induces a kernel:

$$
K(x,x') = \sum_i \frac{\partial f(x)}{\partial \theta_i}\frac{\partial f(x')}{\partial \theta_i}
$$

This connects QNNs to kernel methods (QSVM, see below) and allows the training dynamics to be analyzed analytically.


## Implementing QNNs with PennyLane

PennyLane provides QNodes, differentiable quantum functions that
can be integrated with Python ML frameworks.  Here we illustrate
building and training a simple variational quantum classifier using
PennyLane.

In [1]:
import pennylane as qml
from pennylane import numpy as np

# Create a 2-qubit simulator device
dev = qml.device('default.qubit', wires=2)

# Define a feature map (state preparation) circuit
def feature_map(x):
    qml.RX(x[0], wires=0)
    qml.RX(x[1], wires=1)

# Define a variational (trainable) layer
def variational_layer(params):
    # params is a list of 4 angles for 2 qubits
    qml.Rot(params[0], params[1], params[2], wires=0)
    qml.Rot(params[3], params[0], params[1], wires=1)
    qml.CNOT(wires=[0,1])

# Define the QNode: quantum classifier circuit
@qml.qnode(dev)
def qclassifier(params, x=None):
    # encode data into quantum state
    feature_map(x)
    # apply two variational layers
    variational_layer(params[0:4])
    variational_layer(params[4:8])
    # measure expectation of Z on qubit 0
    return qml.expval(qml.PauliZ(wires=0))

In this code we instantiate a two-qubit device dev.  feature$\_$map(x) encodes the
two-dimensional input x using $R_x$ rotations .
variational$\_$layer(params) is a block of trainable gates (here two Rot
gates and a CNOT).  The @qml.qnode(dev) decorator turns the Python
function qclassifier into a quantum node that returns the expectation
value of $Z$ .  We apply two such layers (with 8 parameters total) to
increase expressivity.

Next, we define a cost function and train:

In [2]:
# Example training data (X: inputs, Y: binary labels {+1,-1})
X = np.array([[0.1, 0.2], [1.5, -0.7], [0.3, 0.8], [0.9, 0.4]])
Y = np.array([1, -1, 1, -1])

# Mean squared error cost
def cost_fn(params):
    preds = [qclassifier(params, x=x) for x in X]
    return np.mean((preds - Y)**2)

# Initialize parameters (8 angles) randomly
init_params = np.random.randn(8, requires_grad=True)

# Choose an optimizer
opt = qml.GradientDescentOptimizer(stepsize=0.1)

# Training loop
params = init_params
for epoch in range(30):
    params = opt.step(cost_fn, params)
    if epoch % 5 == 0:
        loss = cost_fn(params)
        print(f"Epoch {epoch}, loss = {loss:.4f}")

/Users/mhjensen/miniforge3/envs/myenv/lib/python3.9/site-packages/autograd/tracer.py:14: UserWarning: Output seems independent of input.
  warnings.warn("Output seems independent of input.")
/Users/mhjensen/miniforge3/envs/myenv/lib/python3.9/site-packages/autograd/tracer.py:14: UserWarning: Output seems independent of input.
  warnings.warn("Output seems independent of input.")
/Users/mhjensen/miniforge3/envs/myenv/lib/python3.9/site-packages/autograd/tracer.py:14: UserWarning: Output seems independent of input.
  warnings.warn("Output seems independent of input.")
/Users/mhjensen/miniforge3/envs/myenv/lib/python3.9/site-packages/autograd/tracer.py:14: UserWarning: Output seems independent of input.
  warnings.warn("Output seems independent of input.")
/Users/mhjensen/miniforge3/envs/myenv/lib/python3.9/site-packages/autograd/tracer.py:14: UserWarning: Output seems independent of input.
  warnings.warn("Output seems independent of input.")
/Users/mhjensen/miniforge3/envs/myenv/lib/pyt

Epoch 0, loss = 0.5039
Epoch 5, loss = 0.5039
Epoch 10, loss = 0.5039
Epoch 15, loss = 0.5039
Epoch 20, loss = 0.5039
Epoch 25, loss = 0.5039


/Users/mhjensen/miniforge3/envs/myenv/lib/python3.9/site-packages/autograd/tracer.py:14: UserWarning: Output seems independent of input.
  warnings.warn("Output seems independent of input.")
/Users/mhjensen/miniforge3/envs/myenv/lib/python3.9/site-packages/autograd/tracer.py:14: UserWarning: Output seems independent of input.
  warnings.warn("Output seems independent of input.")
/Users/mhjensen/miniforge3/envs/myenv/lib/python3.9/site-packages/autograd/tracer.py:14: UserWarning: Output seems independent of input.
  warnings.warn("Output seems independent of input.")
/Users/mhjensen/miniforge3/envs/myenv/lib/python3.9/site-packages/autograd/tracer.py:14: UserWarning: Output seems independent of input.
  warnings.warn("Output seems independent of input.")
/Users/mhjensen/miniforge3/envs/myenv/lib/python3.9/site-packages/autograd/tracer.py:14: UserWarning: Output seems independent of input.
  warnings.warn("Output seems independent of input.")
/Users/mhjensen/miniforge3/envs/myenv/lib/pyt

This training loop uses PennyLane’s GradientDescentOptimizer which
automatically computes gradients of cost$\_$fn w.r.t. params using the
parameter-shift rule.  One monitors the loss to verify improvement.
In practice, more advanced optimizers (Adam, QNG, etc.) or batch
training may be used.  The printed output shows the loss decreasing
over epochs (assuming a learnable model).

Note: All operations are differentiable because we imported
pennylane.numpy as np.  This ensures that backpropagation through the
QNode and classical operations works seamlessly .

One could also plot the training loss
vs. epochs or the decision boundary learned by the QNN.

## Using PennyLane

PennyLane’s qml.qnode decorator converts a quantum circuit into a
function whose gradients can be computed automatically .  We combine
the feature map and variational layers inside a single QNode to form a
model.  The cost function compares the QNN’s output to labels;
optimization is done classically.  PennyLane allows seamless mixing of
quantum nodes and classical Python code, facilitating experimentation.

## Additional exercises

1. Modify the above code to use qml.AdamOptimizer and compare training convergence.

2. Extend the circuit by adding a third qubit (use wires=3) and corresponding rotations. How does this affect the model’s capacity?

3. Implement a simple dataset (e.g. points arranged in XOR pattern) and train the QNN. Evaluate its classification accuracy.

## Variational Quantum Neural Network for credit classification

This self-contained PennyLane code demonstrates a simple hybrid
quantum-classical binary classifier on synthetic financial data, like
the one we discussed in connection with quantum support vector
machines.

We build a quantum neural network (QNN) – a parameterized
(variational) quantum circuit – to classify synthetic credit data.
Quantum neural networks are typically implemented as variational
quantum circuits with trainable rotation angles.

We first generate
small synthetic data with features like income, debt ratio, and age,
labeling each datapoint as “good” or “bad” credit. Classical features
are encoded into qubit rotation angles using an angle embedding, and a
layer of trainable entangling gates (PennyLane’s
StronglyEntanglingLayers) forms the variational ansatz. During
training, we optimize the circuit parameters via a classical optimizer
to minimize binary cross-entropy loss . Finally, we measure one qubit
to produce a probability for the positive class and evaluate the
classifier with accuracy, precision, and recall . The following code
implements all steps using PennyLane’s default.qubit simulator.

In [ ]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Step 1: Generate synthetic credit data
np.random.seed(0)
N = 100
# Features: income (in thousands), debt_ratio (percent), age (years)
income = np.random.normal(50, 15, N)         # mean 50, std 15
debt_ratio = np.random.uniform(0, 100, N)    # between 0 and 100%
age = np.random.randint(18, 70, N)           # ages 18 to 69
X = np.column_stack((income, debt_ratio, age))

# Label: simple linear rule with noise => 1 = good credit, 0 = bad
score = 0.3 * income - 0.2 * debt_ratio + 0.1 * age
y = (score > np.median(score)).astype(int)   # threshold at median

# Split into train/test (80/20 split)
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: Feature scaling for angle embedding
# Scale each feature to [0, pi] so they can serve as rotation angles
max_income = X[:, 0].max()
max_debt = X[:, 1].max()
min_age, max_age = X[:, 2].min(), X[:, 2].max()

# Scale training data
X_train_scaled = X_train_np.copy()
X_train_scaled[:, 0] = X_train_scaled[:, 0] / max_income * np.pi
X_train_scaled[:, 1] = X_train_scaled[:, 1] / max_debt * np.pi
X_train_scaled[:, 2] = (X_train_scaled[:, 2] - min_age) / (max_age - min_age) * np.pi

# Scale test data
X_test_scaled = X_test_np.copy()
X_test_scaled[:, 0] = X_test_scaled[:, 0] / max_income * np.pi
X_test_scaled[:, 1] = X_test_scaled[:, 1] / max_debt * np.pi
X_test_scaled[:, 2] = (X_test_scaled[:, 2] - min_age) / (max_age - min_age) * np.pi

# Convert data to PennyLane numpy arrays for differentiation
X_train = pnp.array(X_train_scaled)
X_test  = pnp.array(X_test_scaled)
y_train = pnp.array(y_train_np)
y_test  = pnp.array(y_test_np)

# Step 3: Define the variational quantum circuit
n_qubits = 3
dev = qml.device("default.qubit", wires=n_qubits)

# Parameterized quantum neural network (variational circuit)
@qml.qnode(dev)
def circuit(weights, x):
    # Feature map: encode features by rotation angles on each qubit
    # Uses RY rotations (AngleEmbedding by default uses RX or can specify RY)
    qml.AngleEmbedding(features=x, wires=range(n_qubits), rotation='Y')
    # Variational (trainable) layers: strong entangling rotations
    qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    # Measure expectation of Pauli-Z on the first qubit
    return qml.expval(qml.PauliZ(0))

# Initialize trainable weights for the variational layers
num_layers = 1
# Shape for StronglyEntanglingLayers: (num_layers, n_qubits, 3)
init_weights = 0.01 * np.random.randn(num_layers, n_qubits, 3)
weights = pnp.array(init_weights, requires_grad=True)

# Step 4: Define cost (binary cross-entropy) and train the QNN
def cross_entropy_loss(weights, X, y):
    # Run circuit on each sample to get expectation values
    expvals = [circuit(weights, x=x) for x in X]
    expvals = pnp.stack(expvals)
    # Convert expectation ⟨Z⟩ to probability for label=1: P(1) = (1 - ⟨Z⟩)/2
    probs = (1 - expvals) / 2
    # Clip probabilities to avoid log(0)
    probs = pnp.clip(probs, 1e-6, 1 - 1e-6)
    # Binary cross-entropy loss
    loss = -pnp.mean(y * pnp.log(probs) + (1 - y) * pnp.log(1 - probs))
    return loss

# Choose an optimizer (gradient descent)
opt = qml.GradientDescentOptimizer(stepsize=0.5)

# Training loop
epochs = 20
for it in range(epochs):
    weights, cost_val = opt.step_and_cost(lambda w: cross_entropy_loss(w, X_train, y_train), weights)
    if (it + 1) % 5 == 0:
        print(f"Iteration {it+1:>2}: loss = {cost_val:.4f}")

# Step 5: Evaluate performance on training and test sets
# Predict by evaluating circuit and thresholding at 0.5
def predict(weights, X):
    preds = []
    for x in X:
        z = circuit(weights, x=x)
        prob = float((1 - z) / 2)  # probability of class=1
        preds.append(int(prob > 0.5))
    return np.array(preds)

y_train_pred = predict(weights, X_train)
y_test_pred  = predict(weights, X_test)

# Compute accuracy, precision, recall
train_acc = accuracy_score(y_train_np, y_train_pred)
test_acc  = accuracy_score(y_test_np, y_test_pred)
train_prec = precision_score(y_train_np, y_train_pred)
test_prec  = precision_score(y_test_np, y_test_pred)
train_rec  = recall_score(y_train_np, y_train_pred)
test_rec   = recall_score(y_test_np, y_test_pred)

print(f"Train Accuracy:  {train_acc:.2f}, Precision: {train_prec:.2f}, Recall: {train_rec:.2f}")
print(f"Test  Accuracy:  {test_acc:.2f}, Precision: {test_prec:.2f}, Recall: {test_rec:.2f}")

Iteration  5: loss = 0.6141
Iteration 10: loss = 0.5260
Iteration 15: loss = 0.4992


## Essential steps in the code
**Data Encoding:**

We map each feature vector to quantum states via angle embedding: each
feature is used as the rotation angle of an RY gate on a qubit . This
creates a **quantum feature map** of our classical data.

**Variational Ansatz:**

After embedding, we apply a layer of trainable rotations and
entangling gates (StronglyEntanglingLayers), creating a parameterized
circuit whose outputs depend on adjustable weights . Measuring the
expectation $\langle Z\rangle$ of the first qubit yields a value in $[-1,1]$, which we
convert to a class probability via $(1–\langle Z\rangle)/2$.

**Training:**

We optimize the circuit parameters by minimizing the binary
cross-entropy loss between the predicted probabilities and true
labels. Binary cross-entropy (log-loss) is a standard choice for
binary classification , adjusting weights to improve the match between
predictions and targets. We use PennyLane’s GradientDescentOptimizer
(or AdamOptimizer) to update parameters via backpropagated gradients.

**Evaluation:**

Finally, we compute accuracy, precision, and recall on the
dataset. Accuracy is the fraction of correct predictions. Precision is
the fraction of predicted “good” credits that are truly good, and
recall is the fraction of actual good credits that are correctly
identified. These metrics are standard in classification tasks .

---
## Variational Quantum Eigensolver (VQE)

*VQE is the premier near-term quantum algorithm for ground-state energy estimation. It directly applies the variational principle on a quantum device.*

## Variational Principle and Ansatz

Given a Hamiltonian $H$, the exact ground-state energy satisfies:

$$
E_0 = \min_{\psi}\langle\psi|H|\psi\rangle
$$

For any trial state $|\psi(\theta)\rangle = U(\theta)|0\rangle$ with $U(\theta) = e^{-iH(\theta)}$, $H(\theta) = \sum_\alpha \theta_\alpha P_\alpha$:

$$
E(\theta) = \langle\psi(\theta)|H|\psi(\theta)\rangle \geq E_0
$$

The variational upper bound is minimized classically over $\theta$.

### Hamiltonian Decomposition and Measurement

Any qubit Hamiltonian decomposes into Pauli strings:

$$
H = \sum_i c_i P_i, \qquad
E(\theta) = \sum_i c_i \langle P_i \rangle
$$

Each $\langle P_i\rangle$ is measured separately on the quantum device; the classical computer accumulates the sum.


## VQE Gradient and Parameter Shift

The gradient of the energy has the same structure as for QNNs:

$$
\frac{\partial E}{\partial \theta} = i\langle\psi|[H_\theta, H]|\psi\rangle
$$

and is evaluated exactly by the **parameter-shift rule**:

$$
\frac{\partial E}{\partial \theta} = \frac{1}{2}\left[E\!\left(\theta+\tfrac{\pi}{2}\right) - E\!\left(\theta-\tfrac{\pi}{2}\right)\right]
$$

The connection to TDVP is direct: VQE optimization *is* projected quantum dynamics,

$$
\sum_j F_{ij}\,\dot{\theta}_j = C_i
$$

where $F_{ij}$ is the QFI and $C_i = \mathrm{Im}\langle\partial_i\psi|H|\psi\rangle$.

### Unitary Coupled Cluster (UCC) Ansatz

The physically motivated UCC ansatz is:

$$
U = e^{T - T^\dagger}, \qquad T = \sum_{ai} t_{ai}\,a_a^\dagger a_i + \cdots
$$

Implemented via **Trotterization**: $e^{A+B} \approx e^A e^B$.
The BCH expansion connects UCC directly to coupled-cluster theory:

$$
e^{-T}He^T = H + [H,T] + \tfrac{1}{2}[[H,T],T] + \cdots
$$

**Summary:** VQE = variational ground-state solver; connected to QNN via the same gradient and geometry; strong link to many-body physics.


---
## Quantum Approximate Optimization Algorithm (QAOA)

*QAOA is the leading near-term algorithm for combinatorial optimization. It is simultaneously a special-case QNN, a discretized adiabatic evolution, and a Trotterized Hamiltonian simulation.*

## QAOA: Problem Statement and Ansatz

**Goal:** Maximize (or minimize) a classical cost function $C(z)$ over bit strings $z \in \{0,1\}^n$.

Encode the cost as a diagonal Hamiltonian:

$$
H_C\,|z\rangle = C(z)\,|z\rangle
$$

The **QAOA ansatz** of depth $p$ is:

$$
|\gamma,\beta\rangle = \prod_{l=1}^{p} e^{-i\beta_l H_M}\,e^{-i\gamma_l H_C}\,|+\rangle
$$

where:

- $H_C$: **cost Hamiltonian** (problem-specific)
- $H_M = \sum_i X_i$: **mixer Hamiltonian** (drives transitions between bit strings)
- $|+\rangle = H^{\otimes n}|0\rangle$: uniform superposition initial state
- $\gamma = (\gamma_1,\ldots,\gamma_p)$, $\beta = (\beta_1,\ldots,\beta_p)$: **variational parameters**

The objective is to maximize:

$$
C(\gamma,\beta) = \langle\gamma,\beta|H_C|\gamma,\beta\rangle
$$


## QAOA: Adiabatic Motivation and Trotter Connection

QAOA is best understood as a **discretization of adiabatic quantum computation**.
The adiabatic interpolation is:

$$
H(s) = (1-s)H_M + s H_C, \qquad s \in [0,1]
$$

Adiabatic theorem: if this is swept slowly, the system stays in the ground state and reaches the ground state of $H_C$ (the optimal solution).

**Trotterization** of the time-evolution operator gives exactly the QAOA circuit:

$$
e^{-i(H_M + H_C)t} \approx e^{-iH_M\Delta t}\,e^{-iH_C\Delta t}
$$

So QAOA at depth $p$ is a $p$-step Trotterized adiabatic evolution with the step sizes as variational parameters — and in the limit $p \to \infty$, QAOA recovers exact adiabatic evolution.

### MaxCut: The Canonical Example

For the **MaxCut problem** on graph $G=(V,E)$, the cost Hamiltonian is:

$$
H_C = \sum_{\langle ij\rangle \in E} \frac{1 - Z_i Z_j}{2}
$$

The QAOA alternates between:
1. **Phase oracle** $e^{-i\gamma H_C}$: encodes the graph cut structure
2. **Mixer** $e^{-i\beta H_M}$: creates superpositions and explores the solution space


## QAOA: Gradient, Landscape, and Relations

**Gradient** of the cost with respect to $\gamma_l$:

$$
\partial_{\gamma_l} C = i\langle [H_C, H_{\mathrm{eff}}] \rangle
$$

The **parameter landscape** $C(\gamma,\beta)$ is non-convex but has structured correlations exploitable by warm-starting and parameter concentration.

### QAOA as a QNN

QAOA is a **QNN with specific Hamiltonians** $H_C$ and $H_M$ — it is a structured ansatz rather than a hardware-efficient one. This makes it:

- More problem-specific and physically motivated
- Easier to analyze (known Hamiltonian structure)
- Less expressive than general VQE but more efficient for combinatorial problems

| Feature | QAOA | VQE |
|---|---|---|
| Ansatz | Fixed ($H_C, H_M$) | General Pauli sum |
| Motivation | Adiabatic / Trotter | Variational principle |
| Target | Combinatorial optim. | Ground-state energy |
| Parameters | $2p$ ($\gamma,\beta$) | Many ($\theta_\alpha$) |

### Key References and Resources

- **Original paper:** Farhi, Goldstone, Gutmann, [arXiv:1411.4028](https://arxiv.org/abs/1411.4028) (2014)
- **Review:** Blekos et al., *A Review on Quantum Approximate Optimization Algorithm and its Variants*, [arXiv:2306.09198](https://arxiv.org/abs/2306.09198) (2023)
- **PennyLane QAOA tutorial:** [pennylane.ai/qml/demos/tutorial_qaoa_intro](https://pennylane.ai/qml/demos/tutorial_qaoa_intro/)
- **Qiskit QAOA tutorial:** [learning.quantum.ibm.com/tutorial/quantum-approximate-optimization-algorithm](https://learning.quantum.ibm.com/tutorial/quantum-approximate-optimization-algorithm)
- **IBM Quantum / Qiskit:** [QAOA in Qiskit](https://qiskit-community.github.io/qiskit-algorithms/stubs/qiskit_algorithms.QAOA.html)
- **Landscape and performance:** Streif & Leib, [arXiv:1904.02042](https://arxiv.org/abs/1904.02042); Zhou et al., [arXiv:1812.01041](https://arxiv.org/abs/1812.01041)


In [ ]:
"""
QAOA for MaxCut on a small graph — PennyLane implementation.

The MaxCut cost Hamiltonian for edge (i,j) is  (1 - Z_i Z_j)/2.
The QAOA circuit alternates cost and mixer layers, parameterized by
vectors gamma (cost angles) and beta (mixer angles).
"""
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt

# ── Problem: MaxCut on a 4-node ring graph ──────────────────────
edges = [(0,1), (1,2), (2,3), (3,0)]
n_qubits = 4
p = 2        # QAOA depth

# Build cost Hamiltonian  H_C = sum_{(i,j)} (1 - Z_i Z_j) / 2
def cost_hamiltonian(edges, n):
    coeffs, obs = [], []
    for i, j in edges:
        coeffs.append(-0.5)          # -1/2 * Z_i Z_j  (+ const absorbed)
        obs.append(qml.PauliZ(i) @ qml.PauliZ(j))
    return qml.Hamiltonian(coeffs, obs)

H_C = cost_hamiltonian(edges, n_qubits)

# ── QAOA circuit ─────────────────────────────────────────────────
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def qaoa_circuit(params):
    gamma = params[:p]     # cost angles
    beta  = params[p:]     # mixer angles

    # Initial state: uniform superposition
    for i in range(n_qubits):
        qml.Hadamard(wires=i)

    for layer in range(p):
        # Cost unitary  exp(-i gamma H_C)
        for i, j in edges:
            qml.CNOT(wires=[i, j])
            qml.RZ(2 * gamma[layer], wires=j)
            qml.CNOT(wires=[i, j])
        # Mixer unitary  exp(-i beta H_M),  H_M = sum_i X_i
        for i in range(n_qubits):
            qml.RX(2 * beta[layer], wires=i)

    return qml.expval(H_C)

# ── Optimize ─────────────────────────────────────────────────────
def cost_fn(params):
    return qaoa_circuit(params)     # maximise = minimise negative

opt = qml.AdamOptimizer(stepsize=0.1)
params = np.array([0.5, 0.3, 0.4, 0.2], requires_grad=True)

costs = []
for step in range(80):
    params, val = opt.step_and_cost(cost_fn, params)
    costs.append(float(val))
    if (step + 1) % 20 == 0:
        print(f"Step {step+1:3d}:  <H_C> = {val:.4f}")

# ── Plot convergence ─────────────────────────────────────────────
plt.figure(figsize=(7, 3))
plt.plot(costs)
plt.xlabel("Optimization step"); plt.ylabel(r"$\langle H_C \rangle$")
plt.title("QAOA MaxCut convergence (p=2, 4-node ring)")
plt.tight_layout(); plt.show()

# ── Sample the best bit string ────────────────────────────────────
@qml.qnode(dev)
def sample_circuit(params):
    gamma, beta = params[:p], params[p:]
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
    for layer in range(p):
        for i, j in edges:
            qml.CNOT(wires=[i, j]); qml.RZ(2*gamma[layer], wires=j); qml.CNOT(wires=[i, j])
        for i in range(n_qubits):
            qml.RX(2*beta[layer], wires=i)
    return qml.probs(wires=range(n_qubits))

probs = sample_circuit(params)
best  = int(np.argmax(probs))
print(f"\nMost probable bit string: {best:04b}  (probability {float(probs[best]):.3f})")
print("(Optimal MaxCut on a 4-ring: 0101 or 1010 — two independent sets)")


## QAOA Exercises

1. **Depth scaling:** Run the QAOA code above for $p = 1, 2, 3, 4$. Plot $\langle H_C\rangle$ at convergence vs $p$. Does it improve monotonically?

2. **Different graph:** Replace the ring with a complete graph $K_4$ (all 6 edges). What is the optimal MaxCut? Does QAOA find it?

3. **Parameter landscape:** Fix $p=1$ and scan $C(\gamma, \beta)$ over a 2D grid. Plot the landscape. Identify local minima.

4. **Mixer variation:** Replace the transverse-field mixer $\sum_i X_i$ with the XY mixer $\sum_{\langle ij\rangle}(X_iX_j + Y_iY_j)$. Discuss the effect on the feasible subspace.

5. **Warm starting:** Initialize $\gamma, \beta$ from the $p=1$ optimal parameters when running $p=2$. Compare convergence speed.

6. **Theoretical question:** Show that in the limit $p \to \infty$ with appropriate $\gamma, \beta$ schedules, QAOA recovers adiabatic quantum computation.


---
## Quantum Support Vector Machines (QSVM)

*QSVM uses quantum feature maps to define kernels that are classically hard to evaluate, providing a potential route to quantum advantage in classification.*

## Classical SVM: Primal and Dual

Given labeled data $(x_i, y_i)$ with $y_i \in \{-1,+1\}$, the SVM primal problem is:

$$
\min_{w,b}\,\tfrac{1}{2}\|w\|^2 \quad \text{s.t.} \quad y_i(w\cdot x_i + b) \geq 1
$$

Via Lagrange duality (introducing multipliers $\alpha_i \geq 0$):

$$
\max_\alpha\,\sum_i \alpha_i - \tfrac{1}{2}\sum_{ij}\alpha_i\alpha_j y_i y_j (x_i\cdot x_j)
$$

The **kernel trick** replaces inner products $x_i \cdot x_j \to K(x_i, x_j) = \langle\phi(x_i),\phi(x_j)\rangle$, implicitly embedding data into a high-dimensional feature space.

By the **Mercer theorem**, any positive-definite $K$ admits a spectral decomposition:

$$
K(x,x') = \sum_k \lambda_k\,\psi_k(x)\,\psi_k(x')
$$

defining an embedding into a Reproducing Kernel Hilbert Space (RKHS).

## Quantum Feature Map and Quantum Kernel

Define the quantum feature map:

$$
|\phi(x)\rangle = U(x)|0\rangle, \qquad U(x) = e^{iH(x)}, \qquad H(x) = \sum_i x_i Z_i + \sum_{i<j} x_i x_j Z_i Z_j
$$

The **quantum kernel** is the fidelity between embedded states:

$$
K(x,x') = |\langle\phi(x)|\phi(x')\rangle|^2 = |\langle 0|U^\dagger(x)U(x')|0\rangle|^2
$$

This has a direct physical interpretation: it is a **transition probability** / quantum overlap, analogous to a correlation function in many-body physics.

### Measuring the Kernel: Swap Test and Direct Fidelity

The **swap test** circuit estimates $K$:

$$
P(0) = \frac{1 + |\langle\phi(x)|\phi(x')\rangle|^2}{2}
$$

Alternatively, the **direct fidelity circuit** $U(x)$ then $U^\dagger(x')$ then measure: $P(0) = K(x,x')$.


## QSVM: Expressivity, Concentration, and Connections

The kernel matrix $K_{ij} = K(x_i,x_j)$ is estimated on the quantum device and the SVM dual is solved classically. The decision function is:

$$
f(x) = \sum_i \alpha_i y_i K(x_i, x) + b
$$

### Expressivity vs Entanglement (same trade-off as QNNs)

| Entanglement | Kernel behavior |
|---|---|
| None | Separable kernel — classically simulable |
| Moderate | Rich structure — potential quantum advantage |
| High | Concentration of measure: $K(x,x') \approx 2^{-n}$ — useless |

### Connection to HHL

Kernel methods require solving the linear system $(K + \lambda I)\alpha = y$. The HHL algorithm can solve this in $O(\kappa^2 \log N)$ time, enabling a **fully quantum pipeline**: quantum kernel evaluation + quantum linear solver.

### Connection to QNN via NTK

In the linearized regime, every QNN induces a kernel — the **Quantum Neural Tangent Kernel**:

$$
K_{\mathrm{NTK}}(x,x') = \sum_i \frac{\partial f(x)}{\partial \theta_i}\frac{\partial f(x')}{\partial \theta_i}
$$

So QSVM (explicit kernel) and QNN (implicit kernel via NTK) are two sides of the same coin.


---
## HHL Algorithm: Quantum Linear Systems

*HHL (Harrow–Hassidim–Lloyd) solves $Ax=b$ with exponential speedup over classical methods under certain conditions. It is the quantum analogue of computing a Green's function.*

## HHL Problem Statement and Key Idea

**Problem:** Given Hermitian $A \in \mathbb{C}^{N\times N}$ and efficiently preparable $|b\rangle$, prepare:

$$
|x\rangle \propto A^{-1}|b\rangle
$$

**Spectral decomposition:**

$$
A = \sum_j \lambda_j |u_j\rangle\langle u_j|, \qquad |b\rangle = \sum_j \beta_j|u_j\rangle
\implies A^{-1}|b\rangle = \sum_j \frac{\beta_j}{\lambda_j}|u_j\rangle
$$

HHL implements the spectral transformation $\lambda_j \to \lambda_j^{-1}$ via **Quantum Phase Estimation (QPE)**:

$$
U = e^{iAt}, \quad U|u_j\rangle = e^{i\lambda_j t}|u_j\rangle
\implies |u_j\rangle|0\rangle \xrightarrow{\text{QPE}} |u_j\rangle|\lambda_j\rangle
$$

### Circuit Steps

1. **QPE:** $\sum_j \beta_j |u_j\rangle \to \sum_j \beta_j |u_j\rangle|\lambda_j\rangle$ (encode eigenvalues)
2. **Controlled rotation:** $|\lambda_j\rangle|0\rangle \to |\lambda_j\rangle\!\left(\sqrt{1-C^2/\lambda_j^2}|0\rangle + C/\lambda_j|1\rangle\right)$ (encode $\lambda_j^{-1}$)
3. **Uncompute QPE:** erase eigenvalue register
4. **Post-select** on ancilla $|1\rangle$: leaves $|x\rangle \propto \sum_j \beta_j/\lambda_j|u_j\rangle$

**Complexity:** $\mathcal{O}(\kappa^2 \log N)$ where $\kappa$ = condition number — exponentially faster than classical $O(N\sqrt{\kappa})$ for sparse, well-conditioned systems with efficient state preparation.

### HHL as Green's Function

$$
A^{-1} \sim (\omega I - H)^{-1} = G(\omega)
$$

HHL is precisely the quantum implementation of the **resolvent operator** / **Green's function** of the many-body Hamiltonian $H$. This connects it to spectral weights, response functions, and self-energies in condensed-matter theory.

### Applications in Machine Learning

| ML problem | Linear system |
|---|---|
| Linear regression | $(X^TX)w = X^Ty$ |
| Kernel SVM | $(K + \lambda I)\alpha = y$ |
| Gaussian processes | $K\alpha = y$ |

**Limitations:** The output is a quantum state (not a classical vector); extracting all components requires $O(N)$ measurements, eliminating the speedup for many tasks.


---
## Grand Unification: Quantum Machine Learning as Operator Science

*All QML methods share a common mathematical skeleton: Hilbert space geometry, variational dynamics, and operator inversion.*

## Three Fundamental Structures

$$
\boxed{\text{Quantum Machine Learning} = \text{Geometry} + \text{Dynamics} + \text{Operators}}
$$

| Structure | Mathematical object | QML method |
|---|---|---|
| **Geometry** | QFI metric $F_{ij}$, Hilbert space $\mathcal{H}$ | QSVM (kernel = overlap), QFI |
| **Dynamics** | Variational evolution, TDVP | QNN, VQE, QAOA |
| **Operators** | Inverse maps, Green's functions | HHL, kernel systems |

### Unifying Correspondences

$$
\text{HHL} \leftrightarrow \text{exact inverse operator} \quad K(x,x') = |\langle\phi(x)|\phi(x')\rangle|^2
$$

$$
\text{QSVM} \leftrightarrow \text{kernel geometry} \quad x \to |\phi(x)\rangle \in \mathcal{H}
$$

$$
\text{QNN/VQE/QAOA} \leftrightarrow \text{variational dynamics} \quad \sum_j F_{ij}\dot\theta_j = C_i
$$

$$
\text{Transformers} \leftrightarrow \text{learned operators} \quad u(x) = \sum_{x'}\alpha(x,x')v(x') \approx G(x,x')f(x')
$$

All methods rely on the **spectral structure** $A = \sum_j \lambda_j|u_j\rangle\langle u_j|$:
- HHL transforms eigenvalues
- QNN/VQE learn the eigenstructure
- QSVM measures overlaps

### Learning as Projection

Learning = optimal projection onto a submanifold of Hilbert space:
- **QNN/VQE/QAOA:** project onto variational manifold
- **TDVP:** optimal (QFI-weighted) projection
- **QSVM:** find separating hyperplane in $\mathcal{H}$
- **HHL:** project input onto eigenbasis and apply $\lambda^{-1}$

$$
\boxed{\text{Learning} \equiv \text{Understanding and approximating operators}}
$$


---
## Outlook and Research Directions

## Quantum Advantage: Where Can It Emerge?

The central open question:

$$
\boxed{\text{Where can genuine quantum advantage emerge?}}
$$

Quantum advantage requires either:
- Computational scaling beyond classical methods (e.g., exponential speedup)
- Access to structures difficult to simulate classically (entanglement, interference)

**NISQ device limitations** — noisy, shallow circuits, limited connectivity — restrict circuit depth, exacerbate barren plateaus, and distort gradients: $\langle O\rangle \to \langle O\rangle + \epsilon$.

| Regime | Prospect |
|---|---|
| **Fault-tolerant QC** | HHL, Shor, Grover — provable exponential speedup |
| **NISQ VQA** | VQE, QAOA, QSVM — heuristic, unclear advantage |
| **Quantum kernels** | Advantage only if feature map is classically hard to simulate |

### Expressivity vs Trainability Trade-off

$$
\text{High expressivity (deep circuits)} \quad \longleftrightarrow \quad \text{hard optimization (barren plateaus)}
$$

### Future Directions

- **Fault-tolerant QC:** HHL, QPE, and Shor's algorithm at scale
- **Better ansatz design:** adaptive VQE (ADAPT-VQE), problem-inspired structures
- **Quantum-classical co-design:** tailor circuits to hardware topology
- **Connection to many-body physics:** VQE ↔ coupled cluster, QAOA ↔ adiabatic evolution, HHL ↔ Green's functions
- **Integration with AI architectures:** quantum transformers, quantum reinforcement learning
- **Field-theoretic view:** circuits as discretized quantum field theories; training as renormalization-like flow


## Applications and Examples

### Quantum Classification and Regression
QNNs and QSVMs classify classical data by learning decision boundaries in Hilbert space.
Extensions include multi-class classification, embedding kernels, and hybrid pipelines.

### QAOA for Combinatorial Optimization
QAOA targets NP-hard problems: MaxCut, graph coloring, portfolio optimization, logistics.
See the PennyLane code cell above. Key references:
- [PennyLane QAOA intro](https://pennylane.ai/qml/demos/tutorial_qaoa_intro/)
- [Qiskit QAOA tutorial](https://learning.quantum.ibm.com/tutorial/quantum-approximate-optimization-algorithm)
- Farhi et al., [arXiv:1411.4028](https://arxiv.org/abs/1411.4028)
- Blekos et al. (review), [arXiv:2306.09198](https://arxiv.org/abs/2306.09198)

### Variational Quantum Eigensolver (VQE)
Ground-state energy estimation for quantum chemistry and condensed-matter Hamiltonians.
Uses the same VQC training loop; connects to UCC, coupled-cluster, and TDVP.

### Quantum Generative Models
Variational circuits trained to produce quantum states matching a target distribution:
Quantum GANs, Quantum Boltzmann Machines — active research area.

### Quantum Kernel Methods (QSVM)
VQCs define kernels for classical kernel machines via the fidelity $K(x,x') = |\langle\phi(x)|\phi(x')\rangle|^2$.
PennyLane provides dedicated quantum kernel modules.

### Quantum Reinforcement Learning
QNNs as function approximators (policy or value functions) in RL;
quantum observations embedded into quantum states and trained by classical RL.

### Hardware Demonstrations
Small-scale QML experiments have been run on IBM, Google, and IonQ devices.
See: [IBM Quantum](https://quantum.ibm.com), [Google Quantum AI](https://quantumai.google).


## References

### Quantum Neural Networks and VQC
1. A. Abbas et al., **The power of quantum neural networks**, Nature Comput. Sci. **1**, 403–409 (2021). [DOI](https://doi.org/10.1038/s43588-021-00084-1)
2. E. Anschuetz and B. Kiani, **Quantum variational algorithms are swamped with traps**, Nat. Commun. **13**, 7760 (2022). [arXiv:2205.05786](https://arxiv.org/abs/2205.05786)
3. M. Zhao et al., **A tutorial on quantum machine learning and quantum neural networks**, [arXiv:2504.16131](https://arxiv.org/abs/2504.16131) (2025).
4. PennyLane documentation: <https://pennylane.ai>

### QAOA
5. E. Farhi, J. Goldstone, S. Gutmann, **A quantum approximate optimization algorithm**, [arXiv:1411.4028](https://arxiv.org/abs/1411.4028) (2014).
6. K. Blekos et al., **A review on quantum approximate optimization algorithm and its variants**, Phys. Rep. **1029**, 1–128 (2024). [arXiv:2306.09198](https://arxiv.org/abs/2306.09198)
7. L. Zhou et al., **Quantum approximate optimization algorithm: performance, mechanism, and implementation on near-term devices**, Phys. Rev. X **10**, 021067 (2020). [arXiv:1812.01041](https://arxiv.org/abs/1812.01041)
8. PennyLane QAOA tutorial: <https://pennylane.ai/qml/demos/tutorial_qaoa_intro/>
9. Qiskit QAOA tutorial: <https://learning.quantum.ibm.com/tutorial/quantum-approximate-optimization-algorithm>

### VQE
10. A. Peruzzo et al., **A variational eigenvalue solver on a photonic quantum processor**, Nat. Commun. **5**, 4213 (2014). [arXiv:1304.3061](https://arxiv.org/abs/1304.3061)
11. J. Tilly et al., **The variational quantum eigensolver: a review of methods and best practices**, Phys. Rep. **986**, 1–128 (2022). [arXiv:2111.05176](https://arxiv.org/abs/2111.05176)

### QSVM and Quantum Kernels
12. V. Havlíček et al., **Supervised learning with quantum-enhanced feature spaces**, Nature **567**, 209–212 (2019). [arXiv:1804.11326](https://arxiv.org/abs/1804.11326)
13. M. Schuld and N. Killoran, **Quantum machine learning in feature Hilbert spaces**, Phys. Rev. Lett. **122**, 040504 (2019). [arXiv:1803.07128](https://arxiv.org/abs/1803.07128)

### HHL
14. A. W. Harrow, A. Hassidim, S. Lloyd, **Quantum algorithm for linear systems of equations**, Phys. Rev. Lett. **103**, 150502 (2009). [arXiv:0811.3171](https://arxiv.org/abs/0811.3171)

### Barren Plateaus
15. J. R. McClean et al., **Barren plateaus in quantum neural network training landscapes**, Nat. Commun. **9**, 4812 (2018). [arXiv:1803.11173](https://arxiv.org/abs/1803.11173)

### Information Geometry and TDVP
16. J. Stokes et al., **Quantum natural gradient**, Quantum **4**, 269 (2020). [arXiv:1909.02108](https://arxiv.org/abs/1909.02108)
